In [4]:
import pandas as pd
import numpy as np
import warnings
from bs4 import BeautifulSoup
import os
import pyconll
import re
import emoji
import json
from datasets import load_dataset
from pyarabic.araby import strip_tashkeel, normalize_hamza
from sklearn.model_selection import train_test_split
from camel_tools.dialectid import DialectIdentifier
from transformers import pipeline
warnings.filterwarnings('ignore')

In [5]:
city_to_country = {
    'Aleppo': 'Syria',
    'Alexandria': 'Egypt',
    'Algiers': 'Algeria',
    'Amman': 'Jordan',
    'Aswan': 'Egypt',
    'Baghdad': 'Iraq',
    'Basra': 'Iraq',
    'Beirut': 'Lebanon',
    'Benghazi': 'Libya',
    'Cairo': 'Egypt',
    'Damascus': 'Syria',
    'Doha': 'Qatar',
    'Fes': 'Morocco',
    'Jeddah': 'Saudi Arabia',
    'Jerusalem': 'Palestine',
    'Khartoum': 'Sudan',
    'MSA': 'Modern Standard Arabic',
    'Mosul': 'Iraq',
    'Muscat': 'Oman',
    'Rabat': 'Morocco',
    'Riyadh': 'Saudi Arabia',
    'Salt': 'Jordan',
    'Sanaa': 'Yemen',
    'Sfax': 'Tunisia',
    'Tripoli': 'Libya',
    'Tunis': 'Tunisia'
}


madar_city_map = {
    'KHA': 'Khartoum',
    'TUN': 'Tunis',
    'TRI': 'Tripoli',
    'ALG': 'Algiers',
    'MSA': 'MSA',
    'FES': 'Fes',
    'BEN': 'Benghazi',
    'SAL': 'Salt',
    'JER': 'Jerusalem',
    'BEI': 'Beirut',
    'SFX': 'Sfax',
    'MUS': 'Muscat',
    'MOS': 'Mosul',
    'JED': 'Jeddah',
    'RIY': 'Riyadh',
    'RAB': 'Rabat',
    'DAM': 'Damascus',
    'ASW': 'Aswan',
    'AMM': 'Amman',
    'CAI': 'Cairo',
    'BAG': 'Baghdad',
    'ALE': 'Aleppo',
    'DOH': 'Doha',
    'ALX': 'Alexandria',
    'SAN': 'Sanaa',
    'BAS': 'Basra'
}



identifier = DialectIdentifier().pretrained()

def identify_dialect(texts):
    if isinstance(texts, str):
        return identifier.predict([texts],output='country')[0].top
    if isinstance(texts, list):
        return [i.top for i in identifier.predict(texts,output='country')]
    



model_marb = pipeline('text-classification', model='Ammar-alhaj-ali/arabic-MARBERT-dialect-identification-city', device='cuda')

def identify_dialect_marb(texts):
    if isinstance(texts, str):
        return city_to_country[model_marb([texts], batch_size=32)[0]['label']]
    if isinstance(texts, list):
        return [city_to_country[i['label']] for i in model_marb(texts, batch_size=32, truncation=True, max_length=512)]




model_camel = pipeline('text-classification', model='CAMeL-Lab/bert-base-arabic-camelbert-mix-did-madar-corpus26', device='cuda')

def identify_dialect_camel(texts):
    if isinstance(texts, str):
        return city_to_country[madar_city_map[model_camel([texts], batch_size=32)[0]['label']]]
    if isinstance(texts, list):
        return [city_to_country[madar_city_map[i['label']]] for i in model_camel(texts, batch_size=32, truncation=True, max_length=512)]


In [6]:
def check_for_decripancies(dataset):
    """
    Check for discrepancies between the number of tokens and the number of POS tags in the Shami dataset.
    """
    return [idx for idx,(i,j) in enumerate(zip(dataset['sentence'].tolist(),dataset['POS_Tags'].tolist())) if len(i.split()) != len(j)]


# Part-of-Speech Tags

## Gumar

Gulf Arabic -- Token level

In [6]:
with open('data/Probing/Gumar/TRAIN_annotated_Gumar_corpus.xml', 'r') as f:
    data_1 = f.read()
    bs_data_1 = BeautifulSoup(data_1, 'xml')

with open('data/Probing/Gumar/TEST_annotated_Gumar_corpus.xml', 'r') as f:
    data_2 = f.read()
    bs_data_2 = BeautifulSoup(data_2, 'xml')

with open('data/Probing/Gumar/DEV_annotated_Gumar_corpus.xml', 'r') as f:
    data_3 = f.read()
    bs_data_3 = BeautifulSoup(data_3, 'xml')

gumar = []
for idx,sent in enumerate(bs_data_1.find_all('SENTENCE') + bs_data_2.find_all('SENTENCE') + bs_data_3.find_all('SENTENCE')):
    
    temp = {'Sentence_Id' : idx+1,
            'sentence' : sent.find('FINAL_TEXT').text,
            'Tokens' : [token.text for token in sent.find_all('TOKEN')],
            'POS_Tags' : [token['baseword_pos'] for token in sent.find_all('TOKEN')]}
    gumar.append(temp)

gumar = pd.DataFrame(gumar)
gumar = gumar[gumar['sentence'].str.split().str.len() <= 200]
gumar['Dialect'] = identify_dialect_camel(gumar['sentence'].tolist())
gumar.to_csv("data/Probing/Gumar/Gumar_Cleaned.csv")

print(check_for_decripancies(gumar))
gumar

[]


,Sentence_Id,sentence,Tokens,POS_Tags,Dialect
0,1,سارة : عخير حبيبي,"[سارة, :, عخير, حبيبي]","[NOUN_PROP:FS, PUNC:-, NOUN:MS, NOUN:MS]",Lebanon
1,2,خالد يأشر لها ويقول : اوص اوص . . .,"[خالد, يأشر, لها, ويقول, :, اوص, اوص, ., ., .]","[NOUN_PROP:MS, VERB:I3MS, PREP:-, VERB:I3MS, P...",Saudi Arabia
2,3,غالية : هايات حبيبتي,"[غالية, :, هايات, حبيبتي]","[NOUN_PROP:FS, PUNC:-, INTERJ:-, NOUN:FS]",Tunisia
3,4,محمد فر القير : انا هب جاهل بابا,"[محمد, فر, القير, :, انا, هب, جاهل, بابا]","[NOUN_PROP:MS, VERB:P3MS, NOUN:MS, PUNC:-, PRO...",Iraq
4,5,ناصر : اكيد ما يبى حد يشاركه فيها ولا انت لك ر...,"[ناصر, :, اكيد, ما, يبى, حد, يشاركه, فيها, ولا...","[NOUN_PROP:MS, PUNC:-, ADJ:MS, PART_NEG:-, VER...",Saudi Arabia
...,...,...,...,...,...
15220,15221,ضحك الرجال عليها ، وقالت : هيه كنت اطالع القمر...,"[ضحك, الرجال, عليها, ،, وقالت, :, هيه, كنت, اط...","[VERB:P3MS, NOUN:MS, PREP:-, PUNC:-, VERB:P3FS...",Qatar
15221,15222,جزوي : لا تطنزين هاه . .,"[جزوي, :, لا, تطنزين, هاه, ., .]","[NOUN_PROP:FS, PUNC:-, PART_NEG:-, VERB:I2FS, ...",Libya
15222,15223,حصة : اسميج انج . . لازم الضمان بعد ؟ ؟,"[حصة, :, اسميج, انج, ., ., لازم, الضمان, بعد, ...","[NOUN_PROP:FS, PUNC:-, VERB:I1S, VERB_PSEUDO:-...",Iraq
15223,15224,سارة : هيه,"[سارة, :, هيه]","[NOUN_PROP:FS, PUNC:-, INTERJ:-]",Jordan


In [7]:
gumar['Dialect'].value_counts()

Dialect
Qatar                     5305
Saudi Arabia              1774
Oman                      1633
Iraq                      1430
Yemen                     1184
Libya                      873
Tunisia                    720
Syria                      656
Jordan                     562
Lebanon                    359
Morocco                    235
Algeria                    146
Sudan                      120
Egypt                      108
Palestine                   65
Modern Standard Arabic      35
Name: count, dtype: int64

In [8]:
# Saudi 
if not os.path.exists('data_dialects/Probing/POS/Saudi/Gumar'):
    os.mkdir('data_dialects/Probing/POS/Saudi/Gumar')
gumar_saudi = gumar[gumar['Dialect'] == 'Saudi Arabia']
train, test = train_test_split(gumar_saudi, test_size=0.2, random_state=42)
with open('data_dialects/Probing/POS/Saudi/Gumar/gumar_saudi_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/Saudi/Gumar/gumar_saudi_label_train.txt', 'w') as file:
    for tags in train['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/POS/Saudi/Gumar/gumar_saudi_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/Saudi/Gumar/gumar_saudi_label_test.txt', 'w') as file:
    for tags in test['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")

# Oman
if not os.path.exists('data_dialects/Probing/POS/Oman/Gumar'):
    os.mkdir('data_dialects/Probing/POS/Oman/Gumar')
gumar_oman = gumar[gumar['Dialect'] == 'Oman']
train, test = train_test_split(gumar_oman, test_size=0.2, random_state=42)
with open('data_dialects/Probing/POS/Oman/Gumar/gumar_oman_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/Oman/Gumar/gumar_oman_label_train.txt', 'w') as file:
    for tags in train['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/POS/Oman/Gumar/gumar_oman_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/Oman/Gumar/gumar_oman_label_test.txt', 'w') as file:
    for tags in test['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")

## QCRI Arabic Dialect

Egypt, Gulf, Leventine, Magrabi -- Token level

In [9]:
with open('data/Probing/QCRI_Arabic_Dialect/seg_plus_pos_egy.txt', 'r') as f:
    data_1 = f.read()
    data_1 = pd.DataFrame([{i:j for i,j in zip(data_1.split('\n')[0].split('\t'),row.split('\t'))} for row in data_1.split('\n')[1:]])
    data_1.dropna(subset=['Word'], inplace=True)
    
    data_1_df = []

    for sent_id in data_1['SentID'].unique():
        
        df = data_1[data_1['SentID'] == sent_id]
        df = df[(df['Word'] != 'EOS') & (df['Word'] != '')]
        temp = {'Sentence_ID' : sent_id,
                'sentence' : ' '.join(df['Word'].tolist()),
                'Tokens' : df['Word'].tolist(),
                'POS_Tags' : df['POS'].tolist()}
        
        data_1_df.append(temp)
    
    data_1_df = pd.DataFrame(data_1_df)
    data_1_df['Dialect'] = 'EGY'

with open('data/Probing/QCRI_Arabic_Dialect/seg_plus_pos_glf.txt', 'r') as f:
    data_2 = f.read()
    data_2 = pd.DataFrame([{i:j for i,j in zip(data_2.split('\n')[0].split('\t'),row.split('\t'))} for row in data_2.split('\n')[1:]])
    data_2.dropna(subset=['Word'], inplace=True)
    
    data_2_df = []

    for sent_id in data_2['SentID'].unique():
        
        df = data_2[data_2['SentID'] == sent_id]
        df = df[(df['Word'] != 'EOS') & (df['Word'] != '')]
        temp = {'Sentence_ID' : sent_id,
                'sentence' : ' '.join(df['Word'].tolist()),
                'Tokens' : df['Word'].tolist(),
                'POS_Tags' : df['POS'].tolist()}
        
        data_2_df.append(temp)
    
    data_2_df = pd.DataFrame(data_2_df)
    data_2_df['Dialect'] = 'GLF'

with open('data/Probing/QCRI_Arabic_Dialect/seg_plus_pos_lev.txt', 'r') as f:
    data_3 = f.read()
    data_3 = pd.DataFrame([{i:j for i,j in zip(data_3.split('\n')[0].split('\t'),row.split('\t'))} for row in data_3.split('\n')[1:]])
    data_3.dropna(subset=['Word'], inplace=True)
    
    data_3_df = []

    for sent_id in data_3['SentID'].unique():
        
        df = data_3[data_3['SentID'] == sent_id]
        df = df[(df['Word'] != 'EOS') & (df['Word'] != '')]
        temp = {'Sentence_ID' : sent_id,
                'sentence' : ' '.join(df['Word'].tolist()),
                'Tokens' : df['Word'].tolist(),
                'POS_Tags' : df['POS'].tolist()}
        
        data_3_df.append(temp)
    
    data_3_df = pd.DataFrame(data_3_df)
    data_3_df['Dialect'] = 'LEV'

with open('data/Probing/QCRI_Arabic_Dialect/seg_plus_pos_mgr.txt', 'r') as f:
    data_4 = f.read()
    data_4 = pd.DataFrame([{i:j for i,j in zip(data_4.split('\n')[0].split('\t'),row.split('\t'))} for row in data_4.split('\n')[1:]])
    data_4.dropna(subset=['Word'], inplace=True)

    data_4_df = []

    for sent_id in data_4['SentID'].unique():
        
        df = data_4[data_4['SentID'] == sent_id]
        df = df[(df['Word'] != 'EOS') & (df['Word'] != '')]
        temp = {'Sentence_ID' : sent_id,
                'sentence' : ' '.join(df['Word'].tolist()),
                'Tokens' : df['Word'].tolist(),
                'POS_Tags' : df['POS'].tolist()}
        
        data_4_df.append(temp)
    
    data_4_df = pd.DataFrame(data_4_df)
    data_4_df['Dialect'] = 'MGR'

data_1_df['Dialect'] = identify_dialect_camel(data_1_df['sentence'].tolist())
data_2_df['Dialect'] = identify_dialect_camel(data_2_df['sentence'].tolist())
data_3_df['Dialect'] = identify_dialect_camel(data_3_df['sentence'].tolist())
data_4_df['Dialect'] = identify_dialect_camel(data_4_df['sentence'].tolist())

qcri = pd.concat([data_1_df,data_2_df,data_3_df,data_4_df])
qcri.to_csv('data/Probing/QCRI_Arabic_Dialect/QCRI_Arabic_Dialect_Cleaned.csv')

print(check_for_decripancies(qcri))
qcri

[]


,Sentence_ID,sentence,Tokens,POS_Tags,Dialect
0,1,ليه لما تحب حد من قلبك يطلع واطى ليه لما تلعب ...,"[ليه, لما, تحب, حد, من, قلبك, يطلع, واطى, ليه,...","[PART, PART, V, NOUN, PREP, NOUN+PRON, V, ADJ,...",Egypt
1,2,"عارف بيقولك ايه "" إذا أخطأت فأحسن "" . . يعني م...","[عارف, بيقولك, ايه, "", إذا, أخطأت, فأحسن, "", ....","[ADJ, PROG_PART+V+PREP+PRON, PART, PUNC, PART,...",Egypt
2,3,الحمد لله يا جدعان الفرسان اللي اتمسكوا عند ست...,"[الحمد, لله, يا, جدعان, الفرسان, اللي, اتمسكوا...","[DET+NOUN, PREP+NOUN, PART, NOUN, DET+NOUN, PA...",Egypt
3,4,بحس بشخصيتي القوية لما اقول لاخويا اعمل حاجة ....,"[بحس, بشخصيتي, القوية, لما, اقول, لاخويا, اعمل...","[PROG_PART+V, PREP+NOUN+NSUFF+PRON, DET+ADJ+NS...",Sudan
4,5,@ahmedabodsheesh يا باشا دي مش محتاجه دراسه دي...,"[@ahmedabodsheesh, يا, باشا, دي, مش, محتاجه, د...","[MENTION, PART, NOUN, PRON, PART, ADJ+NSUFF, N...",Egypt
...,...,...,...,...,...
345,346,@Minocha09459112 @fati_tanjawia يرحم باباك حبس...,"[@Minocha09459112, @fati_tanjawia, يرحم, باباك...","[MENTION, MENTION, V, NOUN+PRON, V+PRON, PART+...",Tunisia
346,347,@besmalg @Melisajodiyaho1 @MinaWeibe @mimita40...,"[@besmalg, @Melisajodiyaho1, @MinaWeibe, @mimi...","[MENTION, MENTION, MENTION, MENTION, MENTION, ...",Algeria
347,348,#SouhilabenLachhab يعجبوني الايهابيين رغم حبهم...,"[#SouhilabenLachhab, يعجبوني, الايهابيين, رغم,...","[HASH, V+PRON+PRON, DET+NOUN+NSUFF, ADV, NOUN+...",Morocco
348,349,يعجبوني الحركى تاع خضراء كي يقولوا فأنز كنزة ه...,"[يعجبوني, الحركى, تاع, خضراء, كي, يقولوا, فأنز...","[V+PRON+PRON, DET+NOUN, PREP, NOUN, PART, V+PR...",Algeria


In [10]:
data_1_df['Dialect'].value_counts(), data_2_df['Dialect'].value_counts(), data_3_df['Dialect'].value_counts(), data_4_df['Dialect'].value_counts()

(Dialect
 Egypt           338
 Libya             5
 Sudan             3
 Yemen             2
 Saudi Arabia      1
 Syria             1
 Name: count, dtype: int64,
 Dialect
 Yemen           103
 Qatar            64
 Saudi Arabia     59
 Iraq             47
 Oman             38
 Egypt             9
 Syria             7
 Jordan            7
 Libya             6
 Sudan             3
 Tunisia           2
 Lebanon           2
 Palestine         1
 Algeria           1
 Morocco           1
 Name: count, dtype: int64,
 Dialect
 Syria           98
 Jordan          82
 Lebanon         75
 Palestine       19
 Egypt           19
 Saudi Arabia    11
 Yemen           11
 Iraq            10
 Oman             9
 Libya            6
 Qatar            6
 Sudan            3
 Tunisia          1
 Name: count, dtype: int64,
 Dialect
 Morocco         145
 Algeria         124
 Tunisia          45
 Libya            13
 Jordan            5
 Oman              5
 Syria             3
 Egypt             3
 Iraq      

In [11]:
# Egypt
if not os.path.exists('data_dialects/Probing/POS/Egypt/QCRI'):
    os.mkdir('data_dialects/Probing/POS/Egypt/QCRI')
qcri_egypt = data_1_df[data_1_df['Dialect'] == 'Egypt']
train, test = train_test_split(qcri_egypt, test_size=0.2, random_state=42)
with open('data_dialects/Probing/POS/Egypt/QCRI/qcri_egypt_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/Egypt/QCRI/qcri_egypt_label_train.txt', 'w') as file:
    for tags in train['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/POS/Egypt/QCRI/qcri_egypt_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/Egypt/QCRI/qcri_egypt_label_test.txt', 'w') as file:
    for tags in test['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")

# Morocco
if not os.path.exists('data_dialects/Probing/POS/Morocco/QCRI'):
    os.mkdir('data_dialects/Probing/POS/Morocco/QCRI')
qcri_morocco = data_4_df[data_4_df['Dialect'] == 'Morocco']
train, test = train_test_split(qcri_morocco, test_size=0.2, random_state=42)
with open('data_dialects/Probing/POS/Morocco/QCRI/qcri_morocco_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/Morocco/QCRI/qcri_morocco_label_train.txt', 'w') as file:
    for tags in train['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/POS/Morocco/QCRI/qcri_morocco_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/Morocco/QCRI/qcri_morocco_label_test.txt', 'w') as file:
    for tags in test['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")


## Camel TB

MSA -- Token level

In [12]:
camel_df = []
counter = 1

for folder in os.listdir('data/Probing/CamelTB'):

    if 'DS_Store' not in folder and 'Cleaned' not in folder and 'camel_sentence' not in folder and 'camel_label' not in folder:
        
        for file_name in os.listdir('data/Probing/CamelTB/' + folder):
            
            file = pyconll.load_from_file('data/Probing/CamelTB/' + folder + '/' + file_name)
            # for sent in file:
            #     temp = {'Sentence_ID' : counter,
            #             'sentence' : sent.text,
            #             'Tokens' : [token.conll().split('\t')[1] for token in sent],
            #             'POS_Tags' : [token.conll().split('\t')[-3] for token in sent]}
            #     camel_df.append(temp)
            #     counter += 1
            for sent in file:
                tokens = {'tokens' : [], 'POS' : []}
                sent_tokens = [token.conll().split('\t')[1] for token in sent]
                pos_tags = [token.conll().split('\t')[3] for token in sent]
                pre_token = ' '
                while len(sent_tokens) > 0:
                    token = sent_tokens.pop(0)
                    pos_tag = pos_tags.pop(0)

                    if pre_token[-1] == '+':
                        tokens['POS'][-1] = tokens['POS'][-1] + '+' + pos_tag
                        tokens['tokens'][-1] = normalize_hamza(strip_tashkeel(tokens['tokens'][-1].replace('+','') + token.replace('+','')))
                    else:
                        if '+' in token:
                            if token[0] == '+':
                                tokens['POS'][-1] = tokens['POS'][-1] + '+' + pos_tag
                                tokens['tokens'][-1] = normalize_hamza(strip_tashkeel(tokens['tokens'][-1].replace('+','') + token.replace('+','')))
                            elif token[-1] == '+':
                                tokens['POS'].append(pos_tag)
                                tokens['tokens'].append(token.replace('+',''))
                        else:
                            tokens['POS'].append(pos_tag)
                            tokens['tokens'].append(token)
                
                temp = {
                    'Sentence_ID' : counter,
                    'sentence' : ' '.join(tokens['tokens']),
                    'Tokens' : tokens['tokens'],
                    'POS_Tags' : tokens['POS']
                }
                camel_df.append(temp)
                counter += 1
                

camel_df = pd.DataFrame(camel_df)
camel_df['Dialect'] = identify_dialect_camel(camel_df['sentence'].tolist())
camel_df.to_csv('data/Probing/CamelTB/Camel_Cleaned.csv')


print(check_for_decripancies(camel_df))
camel_df

[]


,Sentence_ID,sentence,Tokens,POS_Tags,Dialect
0,1,رحلةي إلى بلدي خلال إجازة الحج,"[رحلةي, إلى, بلدي, خلال, إجازة, الحج]","[NOM+NOM, PRT, NOM+NOM, NOM, NOM, NOM]",Oman
1,2,لما وصلت إلى السعودية في هذا الفصل كنت عازما ع...,"[لما, وصلت, إلى, السعودية, في, هذا, الفصل, كنت...","[PRT, VRB, PRT, PROP, PRT, NOM, NOM, VRB, NOM,...",Modern Standard Arabic
2,3,لكنني فوجئت ب خبر محزن و هو ءنه لا يمكن تأجيل ...,"[لكنني, فوجئت, ب, خبر, محزن, و, هو, ءنه, لا, ي...","[PRT+NOM, VRB-PASS, PRT, NOM, NOM, PRT, NOM, P...",Yemen
3,4,إما أن ءلغيه كله ف أرجع في السنة القادمة ل الك...,"[إما, أن, ءلغيه, كله, ف, أرجع, في, السنة, القا...","[PRT, PRT, VRB+NOM, NOM+NOM, PRT, VRB, PRT, NO...",Modern Standard Arabic
4,5,و أيضا أخبرت ب أن زوجةي مقبولة ب جامعة الأميرة...,"[و, أيضا, أخبرت, ب, أن, زوجةي, مقبولة, ب, جامع...","[PRT, NOM, VRB-PASS, PRT, PRT, NOM+NOM, NOM, P...",Saudi Arabia
...,...,...,...,...,...
5750,5751,ل ذلك يجب علىنا أن نسامح بعضنا ل نستطيع العيش ...,"[ل, ذلك, يجب, علىنا, أن, نسامح, بعضنا, ل, نستط...","[PRT, NOM, VRB, PRT+NOM, PRT, VRB, NOM+NOM, PR...",Modern Standard Arabic
5751,5752,الكثير مننا لا يعرف المعنى الحقيقي ل التسامح ف...,"[الكثير, مننا, لا, يعرف, المعنى, الحقيقي, ل, ا...","[NOM, PRT+NOM, PRT, VRB, NOM, NOM, PRT, NOM, P...",Modern Standard Arabic
5752,5753,ل تعزيز هذه الثقافة يمكن أن نعمل على ورشات و م...,"[ل, تعزيز, هذه, الثقافة, يمكن, أن, نعمل, على, ...","[PRT, NOM, NOM, NOM, VRB, PRT, VRB, PRT, NOM, ...",Saudi Arabia
5753,5754,التسامح قيمة عظيمة جدا,"[التسامح, قيمة, عظيمة, جدا]","[NOM, NOM, NOM, NOM]",Modern Standard Arabic


In [13]:
camel_df['Dialect'].value_counts()

Dialect
Modern Standard Arabic    2399
Saudi Arabia              1201
Sudan                      640
Yemen                      399
Oman                       319
Algeria                    268
Tunisia                    110
Morocco                     92
Libya                       83
Iraq                        62
Egypt                       53
Lebanon                     52
Jordan                      32
Palestine                   27
Syria                       15
Qatar                        3
Name: count, dtype: int64

In [14]:
# Modern Standard Arabic
if not os.path.exists('data_dialects/Probing/POS/MSA/Camel'):
    os.mkdir('data_dialects/Probing/POS/MSA/Camel')
camel_msa = camel_df[camel_df['Dialect'] == 'Modern Standard Arabic']
train, test = train_test_split(camel_msa, test_size=0.2, random_state=42)
with open('data_dialects/Probing/POS/MSA/Camel/camel_msa_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/MSA/Camel/camel_msa_label_train.txt', 'w') as file:
    for tags in train['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/POS/MSA/Camel/camel_msa_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/MSA/Camel/camel_msa_label_test.txt', 'w') as file:
    for tags in test['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")

## Shami

Laventine -- Token level

In [15]:
with open('data/Probing/Shami/annotations.json', 'rb') as file:
    shami_corpus = json.load(file)

shami = []
for idx,sent in enumerate(shami_corpus):

    temp = {
        'Sentence_ID' : idx + 1,
        'sentence' : ' '.join([''.join([wordpart['text'] for wordpart in token]) for token in sent['segments']]),
        'Tokens' : [''.join([wordpart['text'] for wordpart in token]) for token in sent['segments']],
        'POS_Tags' : ['+'.join([wordpart['pos'] for wordpart in token]) for token in sent['segments']]
    }
    shami.append(temp)

shami = pd.DataFrame(shami)
shami['Dialect'] = identify_dialect_camel(shami['sentence'].tolist())
shami.to_csv('data/Probing/Shami/Shami_Cleaned.csv')

shami = shami[shami['sentence'].str.split().str.len() == shami['Tokens'].str.len()]
shami.reset_index(drop=True, inplace=True)

print(check_for_decripancies(shami))
shami

[]


,Sentence_ID,sentence,Tokens,POS_Tags,Dialect
0,1,سرعه وموبايل وانعدام المسؤوليه هيدي السواقه بل...,"[سرعه, وموبايل, وانعدام, المسؤوليه, هيدي, السو...","[NOUN, CONJ+NOUN, CONJ+NOUN, PART_DET+NOUN, PR...",Lebanon
1,2,"قلن ما يعلوا الصوت وما يقربوا الكاميرا , اللي ...","[قلن, ما, يعلوا, الصوت, وما, يقربوا, الكاميرا,...","[VERB+PRON, PART_NEG, VERB, PART_DET+NOUN, CON...",Lebanon
2,3,حابب تترك وتمشي,"[حابب, تترك, وتمشي]","[NOUN, VERB, CONJ+VERB]",Syria
3,4,حتى بالدين يعني صار لازم نفصل افلام دينيه مشان...,"[حتى, بالدين, يعني, صار, لازم, نفصل, افلام, دي...","[ADV, PREP+PART_DET+NOUN, VERB, VERB, NOUN, VE...",Syria
4,5,نحنا اللبنانيين اه علقانين بحيرات المجارير او ...,"[نحنا, اللبنانيين, اه, علقانين, بحيرات, المجار...","[PRON, PART_DET+ADJ, INTERJ, NOUN, NOUN, PART_...",Lebanon
...,...,...,...,...,...
1064,1075,شكرا الرئيس تاج راسنا كلنا,"[شكرا, الرئيس, تاج, راسنا, كلنا]","[VERB_NOM, PART_DET+NOUN, NOUN, NOUN+PRON, NOU...",Egypt
1065,1076,قصه يسوع المسيح لا يمكن المس بها بأي شكل كان و...,"[قصه, يسوع, المسيح, لا, يمكن, المس, بها, بأي, ...","[NOUN, NOUN_PROP, PART_DET+NOUN_PROP, PART_NEG...",Yemen
1066,1077,باتمنى اسمع صوت عتويتر من اخو شرموطه معارض بال...,"[باتمنى, اسمع, صوت, عتويتر, من, اخو, شرموطه, م...","[PART+VERB, VERB, NOUN, PREP+FORIEGN, PREP, NO...",Syria
1067,1078,مع جوده ابو خميس,"[مع, جوده, ابو, خميس]","[PREP, NOUN_PROP, NOUN, NOUN_PROP]",Iraq


In [16]:
shami['Dialect'].value_counts()

Dialect
Lebanon                   240
Jordan                    159
Iraq                      144
Syria                     144
Yemen                      93
Libya                      85
Egypt                      37
Oman                       36
Palestine                  35
Tunisia                    28
Saudi Arabia               21
Sudan                      14
Modern Standard Arabic     12
Morocco                     8
Qatar                       7
Algeria                     6
Name: count, dtype: int64

In [17]:
# Jordan
if not os.path.exists('data_dialects/Probing/POS/Jordan/Shami'):
    os.mkdir('data_dialects/Probing/POS/Jordan/Shami')
shami_jordan = shami[shami['Dialect'] == 'Jordan']
train, test = train_test_split(shami_jordan, test_size=0.2, random_state=42)
with open('data_dialects/Probing/POS/Jordan/Shami/shami_jordan_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/Jordan/Shami/shami_jordan_label_train.txt', 'w') as file:
    for tags in train['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/POS/Jordan/Shami/shami_jordan_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/Jordan/Shami/shami_jordan_label_test.txt', 'w') as file:
    for tags in test['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")

# Lebanon
if not os.path.exists('data_dialects/Probing/POS/Lebanon/Shami'):
    os.mkdir('data_dialects/Probing/POS/Lebanon/Shami')
shami_lebanon = shami[shami['Dialect'] == 'Lebanon']
train, test = train_test_split(shami_lebanon, test_size=0.2, random_state=42)
with open('data_dialects/Probing/POS/Lebanon/Shami/shami_lebanon_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/Lebanon/Shami/shami_lebanon_label_train.txt', 'w') as file:
    for tags in train['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/POS/Lebanon/Shami/shami_lebanon_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/Lebanon/Shami/shami_lebanon_label_test.txt', 'w') as file:
    for tags in test['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")

## NArabzi 

Algeria

In [28]:
def read_conllu_narabizi(file_path):

    with open(file_path, 'r') as file:
        data = file.read()
    
    new_data = []
    for sentence in data.strip().split('# sent_id ='):
        temp = {}
        if len(sentence) > 0:
            entire_sentence = ''
            tokens = []
            pos_tags = []
            for token in sentence.split('\n'):
                sentence_id = sentence.split('\n')[0].strip()
                if not token.startswith('#') and not token.strip() == '' and len(token.split('\t')) == 12:
                    token_attributes = token.split('\t')
                    if '-' not in token_attributes[0]:  # Skip multi-word tokens
                        entire_sentence += token_attributes[3] + ' '
                        tokens.append(token_attributes[3])
                        pos_tags.append(token_attributes[5])
            temp['Sentence_ID'] = sentence_id
            temp['sentence'] = entire_sentence.strip()
            temp['Tokens'] = tokens
            temp['POS_Tags'] = pos_tags
            new_data.append(temp)

    new_data = pd.DataFrame(new_data)
    return new_data

data_1 = read_conllu_narabizi('data/Probing/NArabizi/train_NArabizi.conllu')
data_2 = read_conllu_narabizi('data/Probing/NArabizi/dev_NArabizi.conllu')
data_3 = read_conllu_narabizi('data/Probing/NArabizi/test_NArabizi.conllu')

narabizi = pd.concat([data_1, data_2, data_3]).reset_index(drop=True)
narabizi['Dialect'] = identify_dialect_camel(narabizi['sentence'].tolist())
narabizi.to_csv('data/Probing/NArabizi/narabizi_cleaned.csv', index=False)
narabizi

,Sentence_ID,sentence,Tokens,POS_Tags,Dialect
0,M2S30,مام دانجورو كيما البليدة في بارتو,"[مام, دانجورو, كيما, البليدة, في, بارتو]","[ADV, ADJ, ADV, PROPN, ADP, ADV]",Tunisia
1,3100,سلام يا ناس ادعو ربي يسقم احوال ل بلاد قبل و م...,"[سلام, يا, ناس, ادعو, ربي, يسقم, احوال, ل, بلا...","[INTJ, INTJ, NOUN, VERB, NOUN, VERB, NOUN, DET...",Tunisia
2,3015,شعب مريض حتا في لي كمونتير تع الشروق تسبو بعضا...,"[شعب, مريض, حتا, في, لي, كمونتير, تع, الشروق, ...","[NOUN, ADJ, ADV, ADP, DET, NOUN, ADP, PROPN, V...",Algeria
3,4685,قدر الله و ماشاء فعل أعانك الله على بلاؤك,"[قدر, الله, و, ماشاء, فعل, أعانك, الله, على, ب...","[VERB, PROPN, CCONJ, VERB, VERB, VERB, PROPN, ...",Morocco
4,2112,فيف بوقرة نشآلله ال عام ال جاي في ريال مدريد,"[فيف, بوقرة, نشآلله, ال, عام, ال, جاي, في, ريا...","[INTJ, PROPN, INTJ, DET, NOUN, DET, ADJ, ADP, ...",Tunisia
...,...,...,...,...,...
1274,252,زيزو اي تجر لو قرا سيت فوا سي ال بارل اوسي دي ...,"[زيزو, اي, تجر, لو, قرا, سيت, فوا, سي, ال, بار...","[PROPN, VERB, ADV, DET, ADJ, DET, NOUN, ADV, P...",Tunisia
1275,1127,ماتخافوش لي فيناك تاعنا انشالله نفرحو ال امة ا...,"[ماتخافوش, لي, فيناك, تاعنا, انشالله, نفرحو, ا...","[VERB, DET, NOUN, PRON, INTJ, VERB, DET, NOUN,...",Algeria
1276,M2S12,جامي لاشيت نحب هارون رشيد,"[جامي, لاشيت, نحب, هارون, رشيد]","[ADV, VERB, VERB, PROPN, PROPN]",Tunisia
1277,4594,باردو يا شروق امير العار اي نون امير راي,"[باردو, يا, شروق, امير, العار, اي, نون, امير, ...","[INTJ, INTJ, PROPN, NOUN, NOUN, CCONJ, INTJ, N...",Iraq


In [29]:
narabizi['Dialect'].value_counts()

Dialect
Tunisia         731
Morocco         196
Algeria         172
Libya            67
Iraq             45
Lebanon          17
Egypt            11
Sudan            11
Yemen             9
Saudi Arabia      7
Oman              5
Qatar             5
Syria             2
Jordan            1
Name: count, dtype: int64

In [30]:
# Algeria
if not os.path.exists('data_dialects/Probing/POS/Algeria/NArabizi'):
    os.mkdir('data_dialects/Probing/POS/Algeria/NArabizi')
narabizi_algeria = narabizi[narabizi['Dialect'] == 'Algeria']
train, test = train_test_split(narabizi_algeria, test_size=0.2, random_state=42)
with open('data_dialects/Probing/POS/Algeria/NArabizi/narabizi_algeria_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/Algeria/NArabizi/narabizi_algeria_label_train.txt', 'w') as file:
    for tags in train['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/POS/Algeria/NArabizi/narabizi_algeria_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/POS/Algeria/NArabizi/narabizi_algeria_label_test.txt', 'w') as file:
    for tags in test['POS_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")

# Sentiment Analysis

## Algeria YouTube (DZYT)

Algeria

In [6]:
dzyt = load_dataset("Abdou/dz-sentiment-yt-comments")['train'].to_pandas()
dzyt['Sentence_ID'] = range(1, len(dzyt) + 1)
dzyt = dzyt[['Sentence_ID', 'text', 'label']]
dzyt.columns = ['Sentence_ID', 'sentence', 'label']
# replace html tags by '' in text
dzyt['sentence'] = dzyt['sentence'].apply(lambda x: re.sub(r'<.*?>', ' ', x))
# remove urls
dzyt['sentence'] = dzyt['sentence'].apply(lambda x: re.sub(r'http\S+|www\S+|https\S+', ' ', x, flags=re.MULTILINE))
# remove unicode characters and keep only ascii characters and arabic characters
dzyt['sentence'] = dzyt['sentence'].apply(lambda x: re.sub(r'[^\x00-\x7F\u0600-\u06FF]', ' ', x))
# remove emojis
dzyt['sentence'] = dzyt['sentence'].apply(lambda x: emoji.replace_emoji(x, replace=' '))
dzyt['sentence'] = dzyt['sentence'].apply(lambda x: re.sub(' +', ' ', x))
dzyt['sentence'] = dzyt['sentence'].str.strip()

if not os.path.exists('data/Probing/DZYT'):
    os.mkdir('data/Probing/DZYT')
dzyt['Dialect'] = identify_dialect_camel(dzyt['sentence'].tolist())
dzyt = dzyt[dzyt['sentence'] != '']
dzyt.to_csv('data/Probing/DZYT/dzyt_cleaned.csv', index=False)
# Select only negative and positive labels
dzyt = dzyt[dzyt['label'].isin([0, 2])]

In [7]:
dzyt['Dialect'].value_counts()

Dialect
Tunisia                   10256
Algeria                    7828
Morocco                    5202
Saudi Arabia               2759
Libya                      2288
Iraq                       1848
Oman                       1570
Yemen                      1255
Syria                      1159
Lebanon                    1067
Modern Standard Arabic      919
Qatar                       750
Egypt                       688
Sudan                       607
Jordan                      439
Palestine                   193
Name: count, dtype: int64

In [8]:
# Algieria
if not os.path.exists('data_dialects/Probing/Sentiment/Algeria/DZYT'):
    os.mkdir('data_dialects/Probing/Sentiment/Algeria/DZYT')
dzyt_algeria = dzyt[dzyt['Dialect'] == 'Algeria']
train, test = train_test_split(dzyt_algeria, test_size=0.2, random_state=42)
with open('data_dialects/Probing/Sentiment/Algeria/DZYT/dzyt_algeria_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/Algeria/DZYT/dzyt_algeria_label_train.txt', 'w') as file:
    for label in train['label'].tolist():
        file.write(f"{label}\n")
with open('data_dialects/Probing/Sentiment/Algeria/DZYT/dzyt_algeria_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/Algeria/DZYT/dzyt_algeria_label_test.txt', 'w') as file:
    for label in test['label'].tolist():
        file.write(f"{label}\n")

## AET

Egypt

Kora, R., Mohammed, A.: Corpus on Arabic Egyptian Tweets. https://doi.org/10.7910/DVN/LBXV9O

In [11]:
aet = pd.read_excel("data/Probing/AET/40000-Egyptian-tweets.xlsx")
aet['Sentence_ID'] = range(1, len(aet) + 1)
aet = aet[['Sentence_ID', 'review', 'label']]
aet.columns = ['Sentence_ID', 'sentence', 'label']
aet = aet[aet['label'].isin(['positive', 'negative'])]
# replace html tags by '' in text
aet['sentence'] = aet['sentence'].apply(lambda x: re.sub(r'<.*?>', ' ', x))
# remove urls
aet['sentence'] = aet['sentence'].apply(lambda x: re.sub(r'http\S+|www\S+|https\S+', ' ', x, flags=re.MULTILINE))
# remove unicode characters and keep only ascii characters and arabic characters
aet['sentence'] = aet['sentence'].apply(lambda x: re.sub(r'[^\x00-\x7F\u0600-\u06FF]', ' ', x))
# remove emojis
aet['sentence'] = aet['sentence'].apply(lambda x: emoji.replace_emoji(x, replace=' '))
aet['sentence'] = aet['sentence'].apply(lambda x: re.sub(r'\b(POS|NEG|OBJ|NEUTRAL)\b', '', x).strip())
aet['sentence'] = aet['sentence'].apply(lambda x: re.sub(' +', ' ', x))
aet['sentence'] = aet['sentence'].str.strip()
aet = aet[aet['sentence'] != '']

aet['Dialect'] = identify_dialect_camel(aet['sentence'].tolist())

if not os.path.exists('data/Probing/AET'):
    os.mkdir('data/Probing/AET')
aet.to_csv('data/Probing/AET/aet_cleaned.csv', index=False)
aet

,Sentence_ID,sentence,label,Dialect
0,1,اكبر خطا ترتكبه ان تعامل الناس باخلاقك انت مش ...,negative,Libya
1,2,دائما اكره اخر ليله في كل مكان .,negative,Yemen
2,3,يارب اللى يسرق تويتاتى يدخل النار .,negative,Egypt
3,4,الاسراف فى تناول القهوة يسبب الوفاه .,negative,Egypt
4,5,انا اتبهدلت من التراب النهارده. حاجة تقرف .,positive,Egypt
...,...,...,...,...
39995,39996,لنسعد ايامنا بالابتسامة بدلاً ان نملاها بالدموع.,positive,Saudi Arabia
39996,39997,مش هقولك غير ان نص الضحك اللي ضحكته فى حياتي ك...,positive,Egypt
39997,39998,ربنا يوفقك ويسهلك وان شاء الله تعدي الفترة دي ...,positive,Egypt
39998,39999,مبسوطة اوى عملت طريقة مكرونة جديدة و هى دلوقتى...,positive,Egypt


In [12]:
aet['Dialect'].value_counts()

Dialect
Egypt                     24546
Saudi Arabia               3483
Yemen                      2182
Modern Standard Arabic     2180
Libya                      1205
Oman                       1079
Tunisia                     937
Iraq                        859
Sudan                       799
Jordan                      724
Qatar                       551
Algeria                     374
Morocco                     336
Syria                       310
Palestine                   224
Lebanon                     204
Name: count, dtype: int64

In [13]:
# Egypt
if not os.path.exists('data_dialects/Probing/Sentiment/Egypt/AET'):
    os.mkdir('data_dialects/Probing/Sentiment/Egypt/AET')
aet_egypt = aet[aet['Dialect'] == 'Egypt']
train, test = train_test_split(aet_egypt, test_size=0.2, random_state=42)
with open('data_dialects/Probing/Sentiment/Egypt/AET/aet_egypt_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/Egypt/AET/aet_egypt_label_train.txt', 'w') as file:
    for label in train['label'].tolist():
        file.write(f"{label}\n")
with open('data_dialects/Probing/Sentiment/Egypt/AET/aet_egypt_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/Egypt/AET/aet_egypt_label_test.txt', 'w') as file:
    for label in test['label'].tolist():
        file.write(f"{label}\n")

## JHSC

Jordan

In [14]:
df1 = pd.read_csv('data/Probing/JHSC/annotated-hatetweets-4-classes_train.csv')
df2 = pd.read_csv('data/Probing/JHSC/annotated-hatetweets-4-classes_test.csv')

jhsc = pd.concat([df1, df2], ignore_index=True)
jhsc = jhsc[['new_tweet_content', 'Label']]
jhsc['Sentence_ID'] = range(1, len(jhsc) + 1)
jhsc = jhsc[['Sentence_ID', 'new_tweet_content', 'Label']]
jhsc.columns = ['Sentence_ID', 'sentence', 'label']
# replace html tags by '' in text
jhsc['sentence'] = jhsc['sentence'].apply(lambda x: re.sub(r'<.*?>', ' ', x))
# remove urls
jhsc['sentence'] = jhsc['sentence'].apply(lambda x: re.sub(r'http\S+|www\S+|https\S+', ' ', x, flags=re.MULTILINE))
# remove unicode characters and keep only ascii characters and arabic characters
jhsc['sentence'] = jhsc['sentence'].apply(lambda x: re.sub(r'[^\x00-\x7F\u0600-\u06FF]', ' ', x))
# remove emojis
jhsc['sentence'] = jhsc['sentence'].apply(lambda x: emoji.replace_emoji(x, replace=' '))
jhsc['sentence'] = jhsc['sentence'].apply(lambda x: re.sub(r'\b(POS|NEG|OBJ|NEUTRAL)\b', '', x).strip())
jhsc['sentence'] = jhsc['sentence'].apply(lambda x: re.sub(' +', ' ', x))
jhsc['sentence'] = jhsc['sentence'].str.strip()
jhsc['Dialect'] = identify_dialect_camel(jhsc['sentence'].tolist())
jhsc = jhsc[jhsc['sentence'] != '']

if not os.path.exists('data/Probing/JHSC'):
    os.mkdir('data/Probing/JHSC')
jhsc['Dialect'] = identify_dialect_camel(jhsc['sentence'].tolist())
jhsc.to_csv('data/Probing/JHSC/jhsc_cleaned.csv', index=False)
jhsc = jhsc[jhsc['label'].isin(['positive', 'negative'])]
jhsc


,Sentence_ID,sentence,label,Dialect
1,2,اخي العزيز اقول لهولاء الحراميه خذواء المناصب ...,positive,Iraq
2,3,محاضرة الثمنية بتنعس بزيادة,negative,Syria
3,4,الحزن اقرب للإنسان ، الفرح ضيف نوجس منه خيفة .,positive,Tunisia
4,5,الريال يمرض ولا يموت الكبير يظهر في الظروف الص...,positive,Saudi Arabia
5,6,اسعد الله صباحكم بكل خير وبركة وميت وردة لعيون...,negative,Saudi Arabia
...,...,...,...,...
403679,403680,فى شوفتك فرحه تنسيني أحزاني يالي أحبك كثر حب ا...,negative,Yemen
403680,403681,حسبنا الله..سيؤتينا الله من فضله ورسوله...انا ...,negative,Iraq
403684,403685,يحدث في ذكرى يناير يخرج أبناء من السجن ويقتل م...,positive,Tunisia
403685,403686,خليط متناقد من المشاعر مش عارفة ازعل على غيث و...,negative,Saudi Arabia


In [15]:
jhsc['Dialect'].value_counts()

Dialect
Yemen                     31609
Jordan                    30603
Modern Standard Arabic    30527
Saudi Arabia              28163
Oman                      28075
Iraq                      24205
Syria                     18251
Egypt                     16844
Tunisia                   13756
Lebanon                   13198
Libya                     10708
Palestine                  7615
Qatar                      7136
Algeria                    6318
Sudan                      6088
Morocco                    2907
Name: count, dtype: int64

In [16]:
# Jordan
if not os.path.exists('data_dialects/Probing/Sentiment/Jordan/JHSC'):
    os.mkdir('data_dialects/Probing/Sentiment/Jordan/JHSC')
jhsc_jordan = jhsc[jhsc['Dialect'] == 'Jordan']
train, test = train_test_split(jhsc_jordan, test_size=0.2, random_state=42)
with open('data_dialects/Probing/Sentiment/Jordan/JHSC/jhsc_jordan_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/Jordan/JHSC/jhsc_jordan_label_train.txt', 'w') as file:
    for label in train['label'].tolist():
        file.write(f"{label}\n")
with open('data_dialects/Probing/Sentiment/Jordan/JHSC/jhsc_jordan_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/Jordan/JHSC/jhsc_jordan_label_test.txt', 'w') as file:
    for label in test['label'].tolist():
        file.write(f"{label}\n")

## L-HSAB

Lebanon

In [17]:
l_hsab = pd.read_csv('data/Probing/L_HSAB/L-HSAB.txt', sep='\t')
l_hsab['Sentence_ID'] = range(1, len(l_hsab) + 1)
l_hsab = l_hsab[['Sentence_ID', 'Tweet', 'Class']]
l_hsab.columns = ['Sentence_ID', 'sentence', 'label']
# replace html tags by '' in text
l_hsab['sentence'] = l_hsab['sentence'].apply(lambda x: re.sub(r'<.*?>', ' ', x))
# remove urls
l_hsab['sentence'] = l_hsab['sentence'].apply(lambda x: re.sub(r'http\S+|www\S+|https\S+', ' ', x, flags=re.MULTILINE))
# remove unicode characters and keep only ascii characters and arabic characters
l_hsab['sentence'] = l_hsab['sentence'].apply(lambda x: re.sub(r'[^\x00-\x7F\u0600-\u06FF]', ' ', x))
# remove emojis
l_hsab['sentence'] = l_hsab['sentence'].apply(lambda x: emoji.replace_emoji(x, replace=' '))
l_hsab['sentence'] = l_hsab['sentence'].apply(lambda x: re.sub(r'\b(POS|NEG|OBJ|NEUTRAL)\b', '', x).strip())
l_hsab['sentence'] = l_hsab['sentence'].apply(lambda x: re.sub(' +', ' ', x))
l_hsab['sentence'] = l_hsab['sentence'].str.strip()
l_hsab = l_hsab[l_hsab['sentence'] != '']

l_hsab['Dialect'] = identify_dialect_camel(l_hsab['sentence'].tolist())
if not os.path.exists('data/Probing/L_HSAB'):
    os.mkdir('data/Probing/L_HSAB')
l_hsab.to_csv('data/Probing/L_HSAB/l_hsab_cleaned.csv', index=False)
l_hsab = l_hsab[l_hsab['label'].isin(['normal','abusive'])]
l_hsab

,Sentence_ID,sentence,label,Dialect
0,1,الوزير جبران باسيل تاج راسك يا جربان ممنوع بعد...,abusive,Syria
1,2,صديقي انت ابن جامعه اللعبه اكبر من داعش اللعبه...,normal,Iraq
2,3,و مصلحة لبنان تبدأ باستخراج النفط و الغاز لوقف...,normal,Saudi Arabia
3,4,وليد جنبلاط كاتب الحكمة يا قذر,abusive,Jordan
4,5,شو بتلبقلك كلمة خنزير بتجي مفصله على قياسك وشكلك,abusive,Jordan
...,...,...,...,...
5840,5841,اسم الله عليك فرجيني عرض كتافك يا فهيم على فكر...,normal,Jordan
5841,5842,أمير المليشيا مش خائن,normal,Libya
5842,5843,صدقت يناسبك جدا جدا,normal,Yemen
5843,5844,لبخليني حب باسيل شغلتين,normal,Lebanon


In [18]:
l_hsab['Dialect'].value_counts()

Dialect
Lebanon                   1142
Syria                      886
Jordan                     438
Iraq                       419
Modern Standard Arabic     375
Saudi Arabia               332
Yemen                      303
Tunisia                    291
Oman                       274
Libya                      196
Palestine                  193
Egypt                      162
Sudan                      151
Algeria                     96
Qatar                       67
Morocco                     52
Name: count, dtype: int64

In [19]:
# Lebanon
if not os.path.exists('data_dialects/Probing/Sentiment/Lebanon/L_HSAB'):
    os.mkdir('data_dialects/Probing/Sentiment/Lebanon/L_HSAB')
l_hsab_lebanon = l_hsab[l_hsab['Dialect'] == 'Lebanon']
train, test = train_test_split(l_hsab_lebanon, test_size=0.2, random_state=42)
with open('data_dialects/Probing/Sentiment/Lebanon/L_HSAB/l_hsab_lebanon_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/Lebanon/L_HSAB/l_hsab_lebanon_label_train.txt', 'w') as file:
    for label in train['label'].tolist():
        file.write(f"{label}\n")
with open('data_dialects/Probing/Sentiment/Lebanon/L_HSAB/l_hsab_lebanon_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/Lebanon/L_HSAB/l_hsab_lebanon_label_test.txt', 'w') as file:
    for label in test['label'].tolist():
        file.write(f"{label}\n")

## MYC

Morocco

In [20]:
myc = pd.read_excel("data/Probing/MYC/DATA_CLEANED.xlsx")
myc['Sentence_ID'] = range(1, len(myc) + 1)
myc = myc[['Sentence_ID', 'sentence', 'polarity']]
myc.columns = ['Sentence_ID', 'sentence', 'label']
# replace html tags by '' in text
myc['sentence'] = myc['sentence'].apply(lambda x: re.sub(r'<.*?>', ' ', x))
# remove urls
myc['sentence'] = myc['sentence'].apply(lambda x: re.sub(r'http\S+|www\S+|https\S+', ' ', x, flags=re.MULTILINE))
# remove unicode characters and keep only ascii characters and arabic characters
myc['sentence'] = myc['sentence'].apply(lambda x: re.sub(r'[^\x00-\x7F\u0600-\u06FF]', ' ', x))
# remove emojis
myc['sentence'] = myc['sentence'].apply(lambda x: emoji.replace_emoji(x, replace=' '))
myc['sentence'] = myc['sentence'].apply(lambda x: re.sub(' +', ' ', x))
myc['sentence'] = myc['sentence'].str.strip()
myc = myc[myc['sentence'] != '']

if not os.path.exists('data/Probing/MYC'):
    os.mkdir('data/Probing/MYC')
myc['Dialect'] = identify_dialect_camel(myc['sentence'].tolist())
myc.to_csv('data/Probing/MYC/myc_cleaned.csv', index=False)
myc

,Sentence_ID,sentence,label,Dialect
0,1,انسان عبارة عن دواء للإكتئاب,1,Saudi Arabia
1,2,نحبك يا فنان وااااااحسااااااااان,1,Algeria
2,3,Stream zuin Thank you ilyas,1,Iraq
3,4,وحق الرب الى دوا د الاكتئاب الأسطورة,1,Morocco
4,5,اسطورة بكل ما تحمل الكلمة من معنى,1,Saudi Arabia
...,...,...,...,...
19986,19987,"""من هادشي كاااامل اناا ما فهمت والو !!؟؟",-1,Morocco
19987,19988,"""لا حول و قوة إلا بالله العلي العضيم",-1,Saudi Arabia
19988,19989,"""صحافي باش كتحسي من لداخل . كتحس بلخرا اسيدي",-1,Morocco
19989,19990,"""الله يعطيكم الغراق كاملين",-1,Algeria


In [21]:
myc['Dialect'].value_counts()

Dialect
Morocco                   6306
Tunisia                   4264
Algeria                   1857
Saudi Arabia              1209
Iraq                      1136
Lebanon                    790
Libya                      699
Oman                       653
Syria                      577
Yemen                      550
Modern Standard Arabic     345
Egypt                      340
Jordan                     245
Sudan                      207
Qatar                      207
Palestine                   70
Name: count, dtype: int64

In [22]:
# Morocco
if not os.path.exists('data_dialects/Probing/Sentiment/Morocco/MYC'):
    os.mkdir('data_dialects/Probing/Sentiment/Morocco/MYC')
myc_morocco = myc[myc['Dialect'] == 'Morocco']
train, test = train_test_split(myc_morocco, test_size=0.2, random_state=42)
with open('data_dialects/Probing/Sentiment/Morocco/MYC/myc_morocco_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/Morocco/MYC/myc_morocco_label_train.txt', 'w') as file:
    for label in train['label'].tolist():
        file.write(f"{label}\n")
with open('data_dialects/Probing/Sentiment/Morocco/MYC/myc_morocco_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/Morocco/MYC/myc_morocco_label_test.txt', 'w') as file:
    for label in test['label'].tolist():
        file.write(f"{label}\n")

## MARSA

Gulf

In [27]:
marsa = pd.DataFrame()
os.remove('data/Probing/MARSA/marsa_cleaned.csv')
for file in os.listdir('data/Probing/MARSA'):
    df = pd.read_csv(f'data/Probing/MARSA/{file}')
    df = df[['Tweet', 'Polarity']]
    marsa = pd.concat([marsa, df], ignore_index=True)

marsa.columns = ['sentence', 'label']
marsa['Sentence_ID'] = range(1, len(marsa) + 1)
marsa = marsa[['Sentence_ID', 'sentence', 'label']]
marsa['sentence'] = marsa['sentence'].str.strip()
marsa['Dialect'] = identify_dialect_camel(marsa['sentence'].tolist())
marsa.to_csv('data/Probing/MARSA/marsa_cleaned.csv', index=False)
marsa = marsa[marsa['label'].isin(['pos', 'neg'])]

In [28]:
marsa['Dialect'].value_counts()

Dialect
Yemen                     21194
Saudi Arabia              10146
Iraq                       2962
Oman                       2906
Libya                      2118
Egypt                      1545
Qatar                      1145
Tunisia                    1047
Modern Standard Arabic      835
Morocco                     650
Sudan                       352
Lebanon                     232
Algeria                     201
Syria                       154
Jordan                      152
Palestine                    22
Name: count, dtype: int64

In [29]:
# Saudi
if not os.path.exists('data_dialects/Probing/Sentiment/Saudi/MARSA'):
    os.mkdir('data_dialects/Probing/Sentiment/Saudi/MARSA')
marsa_saudi = marsa[marsa['Dialect'] == 'Saudi Arabia']
train, test = train_test_split(marsa_saudi, test_size=0.2, random_state=42)
with open('data_dialects/Probing/Sentiment/Saudi/MARSA/marsa_saudi_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/Saudi/MARSA/marsa_saudi_label_train.txt', 'w') as file:
    for label in train['label'].tolist():
        file.write(f"{label}\n")
with open('data_dialects/Probing/Sentiment/Saudi/MARSA/marsa_saudi_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/Saudi/MARSA/marsa_saudi_label_test.txt', 'w') as file:
    for label in test['label'].tolist():
        file.write(f"{label}\n")

# Oman
if not os.path.exists('data_dialects/Probing/Sentiment/Oman/MARSA'):
    os.mkdir('data_dialects/Probing/Sentiment/Oman/MARSA')
marsa_oman = marsa[marsa['Dialect'] == 'Oman']
train, test = train_test_split(marsa_oman, test_size=0.2, random_state=42)
with open('data_dialects/Probing/Sentiment/Oman/MARSA/marsa_oman_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/Oman/MARSA/marsa_oman_label_train.txt', 'w') as file:
    for label in train['label'].tolist():
        file.write(f"{label}\n")
with open('data_dialects/Probing/Sentiment/Oman/MARSA/marsa_oman_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/Oman/MARSA/marsa_oman_label_test.txt', 'w') as file:
    for label in test['label'].tolist():
        file.write(f"{label}\n")

## LABR

MSA

In [7]:
labr = pd.read_csv("data/Probing/LABR/reviews.tsv", sep="\t", header=None)

labr = labr[[4,0]]
labr.columns = ['sentence', 'label']
labr['Sentence_ID'] = range(1, len(labr) + 1)
labr = labr[['Sentence_ID', 'sentence', 'label']]
labr['sentence'] = labr['sentence'].str.strip()
labr['label']  = labr['label'].map({1: 'negative', 2: 'negative', 3: 'neutral', 4: 'positive', 5: 'positive'})
labr['Dialect'] = identify_dialect_camel(labr['sentence'].tolist())
labr.to_csv('data/Probing/LABR/labr_cleaned.csv', index=False)
labr

,Sentence_ID,sentence,label,Dialect
0,1,"""عزازيل الذي صنعناه ،الكامن في أنفسنا"" يذكرني ...",positive,Modern Standard Arabic
1,2,من أمتع ما قرأت من روايات بلا شك. وحول الشك تد...,positive,Modern Standard Arabic
2,3,رواية تتخذ من التاريخ ،جوًا لها اختار المؤلف ف...,positive,Modern Standard Arabic
3,4,إني أقدّر هذه الرواية كثيرا، لسبب مختلف عن أسب...,negative,Modern Standard Arabic
4,5,الكاهن الذي أطلق على نفسه اسم هيبا تيمنا بالعا...,positive,Modern Standard Arabic
...,...,...,...,...
63252,63253,اجمل مسرحية ألفت في تاريخ الادب الانجليزي,positive,Egypt
63253,63254,بصراحة، لم تكن هذه الرواية على قدر توقعاتي. ال...,neutral,Modern Standard Arabic
63254,63255,هي الرواية الأولى التي قرأتها لأيميلي نصر الله...,neutral,Modern Standard Arabic
63255,63256,تَدْخل بيوت الناس، تدخل قلوبهم، تمدّ يدك تصافح...,neutral,Modern Standard Arabic


In [8]:
labr['Dialect'].value_counts()

Dialect
Modern Standard Arabic    21696
Egypt                     12167
Saudi Arabia              10932
Oman                       6734
Yemen                      4143
Sudan                      3728
Iraq                        971
Algeria                     620
Tunisia                     606
Libya                       567
Jordan                      422
Syria                       213
Morocco                     177
Lebanon                     116
Palestine                    99
Qatar                        66
Name: count, dtype: int64

In [9]:
# Modern Standard Arabic
if not os.path.exists('data_dialects/Probing/Sentiment/MSA/LABR'):
    os.mkdir('data_dialects/Probing/Sentiment/MSA/LABR')
labr_msa = labr[labr['Dialect'] == 'Modern Standard Arabic']
# Balance the dataset
min_count = labr_msa['label'].value_counts().min()
labr_msa_balanced = pd.concat([labr_msa[labr_msa['label'] == label].sample(min_count, random_state=42) for label in labr_msa['label'].unique()])
labr_msa_balanced = labr_msa_balanced.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle the dataset
labr_msa_balanced
train, test = train_test_split(labr_msa_balanced, test_size=0.2, random_state=42)
with open('data_dialects/Probing/Sentiment/MSA/LABR/labr_msa_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/MSA/LABR/labr_msa_label_train.txt', 'w') as file:
    for label in train['label'].tolist():
        file.write(f"{label}\n")
with open('data_dialects/Probing/Sentiment/MSA/LABR/labr_msa_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/Sentiment/MSA/LABR/labr_msa_label_test.txt', 'w') as file:
    for label in test['label'].tolist():
        file.write(f"{label}\n")

# Named Entity Recognition

## DzNER-Corpus

Algeria

In [55]:
dzner = pd.read_csv('data/Probing/DzNER_Corpus/3.NER totale data.csv', header=None)
dzner['sentence_id'] = dzner[0].isna().cumsum()

# Drop NaN rows (sentence separators)
dzner = dzner.dropna(subset=[0, 1])

# Group tokens and labels per sentence
dzner = (
    dzner.groupby('sentence_id')
    .agg({
        0: lambda x: list(x),  # tokens
        1: lambda x: list(x)   # labels
    })
    .reset_index(drop=True)
)

# Rename columns for clarity
dzner.columns = ['Tokens', 'NER_Tags']
dzner['sentence'] = dzner['Tokens'].apply(lambda x: ' '.join(x))

dzner['Sentence_ID'] = range(1, len(dzner) + 1)
dzner = dzner[['Sentence_ID', 'sentence', 'Tokens', 'NER_Tags']]
dzner['Dialect'] = identify_dialect_camel(dzner['sentence'].tolist())
dzner.to_csv('data/Probing/DzNER_Corpus/dzner_cleaned.csv', index=False)
dzner

,Sentence_ID,sentence,Tokens,NER_Tags,Dialect
0,1,كاين ستاظ 1 نوفنبر في باتنة وكاين وهران وعنابة...,"[كاين, ستاظ, 1, نوفنبر, في, باتنة, وكاين, وهرا...","[O, O, O, O, O, B-LOC, O, B-LOC, B-LOC, O, O, ...",Oman
1,2,راهم يعطوهم دراهم الدولة باش يسقمو بصح مشكلة ف...,"[راهم, يعطوهم, دراهم, الدولة, باش, يسقمو, بصح,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Algeria
2,3,بونجاح خارج,"[بونجاح, خارج]","[B-PERS, O]",Lebanon
3,4,هذي هي القوة الضاربة,"[هذي, هي, القوة, الضاربة]","[O, O, O, O]",Saudi Arabia
4,5,لانو تعودنا نسكتو ونخافو ونستعرف بيهم انهم دار...,"[لانو, تعودنا, نسكتو, ونخافو, ونستعرف, بيهم, ا...","[O, O, O, O, O, O, O, O, O, O, O, O]",Libya
...,...,...,...,...,...
5854,5855,شغل بعثة لبيع الاسلحة ماشي سياح نتاوعنا تان يح...,"[شغل, بعثة, لبيع, الاسلحة, ماشي, سياح, نتاوعنا...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Algeria
5855,5856,باسكو باين جايين في إطار صفقة تاع سلاح مع الرو...,"[باسكو, باين, جايين, في, إطار, صفقة, تاع, سلاح...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]",Algeria
5856,5857,المشكل هذا الترويج للسياحة يفهمه الغرب بالعكس ...,"[المشكل, هذا, الترويج, للسياحة, يفهمه, الغرب, ...","[O, O, O, O, O, O, O, O, O, O, O]",Tunisia
5857,5858,خايفين يصرى اي مشكل للسياح الروسيين بسك قادر ي...,"[خايفين, يصرى, اي, مشكل, للسياح, الروسيين, بسك...","[O, O, O, O, O, O, O, O, O, O, O, O, B-LOC]",Sudan


In [56]:
dzner['Dialect'].value_counts()

Dialect
Tunisia                   1719
Algeria                   1153
Morocco                    851
Libya                      385
Iraq                       305
Saudi Arabia               264
Lebanon                    239
Modern Standard Arabic     177
Oman                       159
Yemen                      127
Egypt                      117
Sudan                      117
Syria                      110
Qatar                       61
Jordan                      51
Palestine                   24
Name: count, dtype: int64

In [57]:
# Algeria
if not os.path.exists('data_dialects/Probing/NER/Algeria/DzNER'):
    os.mkdir('data_dialects/Probing/NER/Algeria/DzNER')
dzner_algeria = dzner[dzner['Dialect'] == 'Algeria']
train, test = train_test_split(dzner_algeria, test_size=0.2, random_state=42)
with open('data_dialects/Probing/NER/Algeria/DzNER/dzner_algeria_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/Algeria/DzNER/dzner_algeria_label_train.txt', 'w') as file:
    for tags in train['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/NER/Algeria/DzNER/dzner_algeria_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/Algeria/DzNER/dzner_algeria_label_test.txt', 'w') as file:
    for tags in test['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")

## ACDNER

Egypt

In [58]:
# read json file
with open('data/Probing/ACDNER/egyptian.json') as f:
    data = json.load(f)

rows = []
for sent in data:
    tokens = sent["tokens"]
    tags = ["O"] * len(tokens)
    
    for start, end, label in sent["ner"]:
        tags[start] = f"B-{label}"
        for i in range(start + 1, end + 1):
            tags[i] = f"I-{label}"
    
    rows.append({"Tokens": tokens, "NER_Tags": tags})

acdner_egy = pd.DataFrame(rows)
acdner_egy['sentence'] = acdner_egy['Tokens'].apply(lambda x: ' '.join(x))
acdner_egy['Sentence_ID'] = range(1, len(acdner_egy) + 1)
acdner_egy = acdner_egy[['Sentence_ID', 'sentence', 'Tokens', 'NER_Tags']]
acdner_egy['Dialect'] = identify_dialect_camel(acdner_egy['sentence'].tolist())

In [59]:
acdner_egy['Dialect'].value_counts()

Dialect
Egypt                     210
Jordan                     35
Sudan                      35
Saudi Arabia               22
Modern Standard Arabic     12
Oman                       10
Libya                       9
Algeria                     5
Yemen                       4
Palestine                   4
Lebanon                     2
Morocco                     2
Iraq                        2
Qatar                       1
Name: count, dtype: int64

In [60]:
# Egypt
if not os.path.exists('data_dialects/Probing/NER/Egypt/ACDNER'):
    os.mkdir('data_dialects/Probing/NER/Egypt/ACDNER')
acdner_egypt = acdner_egy[acdner_egy['Dialect'] == 'Egypt']
train, test = train_test_split(acdner_egypt, test_size=0.2, random_state=42)
with open('data_dialects/Probing/NER/Egypt/ACDNER/acdner_egypt_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/Egypt/ACDNER/acdner_egypt_label_train.txt', 'w') as file:
    for tags in train['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/NER/Egypt/ACDNER/acdner_egypt_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/Egypt/ACDNER/acdner_egypt_label_test.txt', 'w') as file:
    for tags in test['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")

## Wojood NER - Curras

Jordan

In [61]:
wojood = pd.concat([pd.read_csv('data/Probing/WojoodNER/train.csv'), pd.read_csv('data/Probing/WojoodNER/test.csv')])
wojood_curras = wojood[wojood['Sub_corpus'] == 'Curras']

all_data = []
for i in wojood_curras['subcorpus_sentence_id'].unique():
    temp = wojood_curras[wojood_curras['subcorpus_sentence_id'] == i]
    temp.sort_values('token_position', inplace=True)
    all_data.append({'Tokens': temp['token'].tolist(), 'NER_Tags': temp['tags'].tolist()})
wojood_curras = pd.DataFrame(all_data)
wojood_curras['sentence'] = wojood_curras['Tokens'].apply(lambda x: ' '.join(x))
wojood_curras['Sentence_ID'] = range(1, len(wojood_curras) + 1)
wojood_curras = wojood_curras[['Sentence_ID', 'sentence', 'Tokens', 'NER_Tags']]
wojood_curras['Dialect'] = identify_dialect_camel(wojood_curras['sentence'].tolist())
wojood_curras.to_csv('data/Probing/WojoodNER/wojood_curras_cleaned.csv', index=False)
wojood_curras

,Sentence_ID,sentence,Tokens,NER_Tags,Dialect
0,1,صفحة يما بديش أتجوز,"[صفحة, يما, بديش, أتجوز]","[O, O, O, O]",Syria
1,2,ما هو حال النت عندكن ? ? ? ?,"[ما, هو, حال, النت, عندكن, ?, ?, ?, ?]","[O, O, O, O, O, O, O, O, O]",Syria
2,3,للازكياء فقط ما هو الجواب ? ? ? ?,"[للازكياء, فقط, ما, هو, الجواب, ?, ?, ?, ?]","[O, O, O, O, O, O, O, O, O]",Saudi Arabia
3,4,اي اكتر شي بتستخدموا لتروحوا على المدرسة,"[اي, اكتر, شي, بتستخدموا, لتروحوا, على, المدرسة]","[O, O, O, O, O, O, O]",Jordan
4,5,او الجامعة ? ? ? ? ?,"[او, الجامعة, ?, ?, ?, ?, ?]","[O, O, O, O, O, O, O]",Qatar
...,...,...,...,...,...
4700,4701,عنا بتطلع بالتكسي . . . ايه ازمة . بتسأل الشوف...,"[عنا, بتطلع, بالتكسي, ., ., ., ايه, ازمة, ., ب...","[O, O, O, O, O, O, O, O, O, O, B-OCC, O, O, O,...",Lebanon
4701,4702,شفيق : شو رأيك يا أستاذ طلعت,"[شفيق, :, شو, رأيك, يا, أستاذ, طلعت]","[O, O, O, O, O, B-OCC, O]",Jordan
4702,4703,طلعت : يبارك فيك استاذ شفيق ( يسلم على الزوجة ),"[طلعت, :, يبارك, فيك, استاذ, شفيق, (, يسلم, عل...","[O, O, O, O, B-OCC, B-PERS, O, O, O, O, O]",Iraq
4703,4704,الرجل يلبس مريول مطبخ ويجلي,"[الرجل, يلبس, مريول, مطبخ, ويجلي]","[O, O, O, O, O]",Tunisia


In [62]:
wojood_curras['Dialect'].value_counts()

Dialect
Jordan                    570
Syria                     520
Saudi Arabia              469
Iraq                      445
Lebanon                   401
Tunisia                   367
Egypt                     343
Yemen                     293
Oman                      235
Libya                     222
Palestine                 190
Modern Standard Arabic    171
Sudan                     152
Algeria                   132
Qatar                     108
Morocco                    87
Name: count, dtype: int64

In [63]:
# Jordan
if not os.path.exists('data_dialects/Probing/NER/Jordan/Wojood'):
    os.mkdir('data_dialects/Probing/NER/Jordan/Wojood')
wojood_jordan = wojood_curras[wojood_curras['Dialect'] == 'Jordan']
train, test = train_test_split(wojood_jordan, test_size=0.2, random_state=42)
with open('data_dialects/Probing/NER/Jordan/Wojood/wojood_jordan_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/Jordan/Wojood/wojood_jordan_label_train.txt', 'w') as file:
    for tags in train['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/NER/Jordan/Wojood/wojood_jordan_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/Jordan/Wojood/wojood_jordan_label_test.txt', 'w') as file:
    for tags in test['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")

## Wojood NER - Baladi

Lebanese

In [64]:
wojood = pd.concat([pd.read_csv('data/Probing/WojoodNER/train.csv'), pd.read_csv('data/Probing/WojoodNER/test.csv')])
wojood_lebanese = wojood[wojood['Sub_corpus'] == 'Lebanese']

all_data = []
for i in wojood_lebanese['subcorpus_sentence_id'].unique():
    temp = wojood_lebanese[wojood_lebanese['subcorpus_sentence_id'] == i]
    temp.sort_values('token_position', inplace=True)
    all_data.append({'Tokens': temp['token'].tolist(), 'NER_Tags': temp['tags'].tolist()})
wojood_lebanese = pd.DataFrame(all_data)
wojood_lebanese['sentence'] = wojood_lebanese['Tokens'].apply(lambda x: ' '.join(x))
wojood_lebanese['Sentence_ID'] = range(1, len(wojood_lebanese) + 1)
wojood_lebanese = wojood_lebanese[['Sentence_ID', 'sentence', 'Tokens', 'NER_Tags']]
wojood_lebanese['Dialect'] = identify_dialect_camel(wojood_lebanese['sentence'].tolist())
wojood_lebanese.to_csv('data/Probing/WojoodNER/wojood_lebanese_cleaned.csv', index=False)
wojood_lebanese

,Sentence_ID,sentence,Tokens,NER_Tags,Dialect
0,1,بين الوعي والّلاوعي عم شوف حلم وانا عا مخدتي م...,"[بين, الوعي, والّلاوعي, عم, شوف, حلم, وانا, عا...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Syria
1,2,لا تفزعي صوب النهر روحي هونيك قلبي ونزفة جروحي...,"[لا, تفزعي, صوب, النهر, روحي, هونيك, قلبي, ونز...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Lebanon
2,3,عنقود الكرم تحلا بكرمه خالقها الله وشو حلوه لم...,"[عنقود, الكرم, تحلا, بكرمه, خالقها, الله, وشو,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]",Yemen
3,4,لو حطو الدنيا حدي وكل الورد علا خدي ما بدي الد...,"[لو, حطو, الدنيا, حدي, وكل, الورد, علا, خدي, م...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Lebanon
4,5,غافلت روحي ورحت شم الورد المزروع بين النهر وعي...,"[غافلت, روحي, ورحت, شم, الورد, المزروع, بين, ا...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Syria
...,...,...,...,...,...
378,379,يتحارب التهرّب الضريبي وتتحسّن جباية يتني الضر...,"[يتحارب, التهرّب, الضريبي, وتتحسّن, جباية, يتن...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Tunisia
379,380,يتزبّط القطاع العام ، يقراها المجالس والوزارات...,"[يتزبّط, القطاع, العام, ،, يقراها, المجالس, وا...","[O, O, O, O, O, O, O, O, O, O, O, O]",Syria
380,381,تتحيّد المْأَسسات التربوية ، وعلى راسا الجامعة...,"[تتحيّد, المْأَسسات, التربوية, ،, وعلى, راسا, ...","[O, O, O, O, O, O, B-ORG, I-ORG, O, O, O, O, O...",Oman
381,382,هالشي بدّو ادارة سليمة لإستخراج المي وتصويب لا...,"[هالشي, بدّو, ادارة, سليمة, لإستخراج, المي, وت...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Syria


In [65]:
wojood_lebanese['Dialect'].value_counts()

Dialect
Lebanon         190
Syria           101
Iraq             18
Tunisia          18
Jordan           13
Palestine         8
Algeria           7
Libya             5
Yemen             5
Egypt             5
Morocco           4
Sudan             4
Oman              3
Qatar             1
Saudi Arabia      1
Name: count, dtype: int64

In [66]:
# Lebanon
if not os.path.exists('data_dialects/Probing/NER/Lebanon/Wojood'):
    os.mkdir('data_dialects/Probing/NER/Lebanon/Wojood')
wojood_lebanon = wojood_lebanese[wojood_lebanese['Dialect'] == 'Lebanon']
train, test = train_test_split(wojood_lebanon, test_size=0.2, random_state=42)
with open('data_dialects/Probing/NER/Lebanon/Wojood/wojood_lebanon_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/Lebanon/Wojood/wojood_lebanon_label_train.txt', 'w') as file:
    for tags in train['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/NER/Lebanon/Wojood/wojood_lebanon_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/Lebanon/Wojood/wojood_lebanon_label_test.txt', 'w') as file:
    for tags in test['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")

## DarNERcorp

Moroccan

In [67]:
dar_ner = pd.concat([pd.read_csv('data/Probing/DarNERcorp/DarNERcorp_train.csv'), pd.read_csv('data/Probing/DarNERcorp/DarNERcorp_test.csv')])
dar_ner = dar_ner.dropna(subset=['Token'])

all_data = []
for i in dar_ner['Sentence'].unique():
    temp = dar_ner[dar_ner['Sentence'] == i]
    all_data.append({'Sentence_ID' : i, 'Tokens': temp['Token'].tolist(), 'NER_Tags': temp['Tag'].tolist()})
dar_ner = pd.DataFrame(all_data)
dar_ner['sentence'] = dar_ner['Tokens'].apply(lambda x: ' '.join(x))
dar_ner = dar_ner[['Sentence_ID', 'sentence', 'Tokens', 'NER_Tags']]
dar_ner['Dialect'] = identify_dialect_camel(dar_ner['sentence'].tolist())
dar_ner.to_csv('data/Probing/DarNERcorp/dar_ner_cleaned.csv', index=False)
dar_ner

,Sentence_ID,sentence,Tokens,NER_Tags,Dialect
0,0,Uppsala ) هيّا رابع أكبر مدينة ف سّويد من بعد ...,"[Uppsala, ), هيّا, رابع, أكبر, مدينة, ف, سّويد...","[B-LOC, O, O, O, O, O, O, B-LOC, O, O, B-LOC, ...",Morocco
1,1,ف 2018 كان عدد سّكان ديالها 172 ، 402 .,"[ف, 2018, كان, عدد, سّكان, ديالها, 172, ،, 402...","[O, O, O, O, O, O, O, O, O, O]",Morocco
2,2,أوپيك ( ب لينݣليزية : OPEC - تاتعني لمنضمة د ل...,"[أوپيك, (, ب, لينݣليزية, :, OPEC, -, تاتعني, ل...","[B-ORG, O, O, B-MISC, O, B-ORG, O, O, B-ORG, I...",Morocco
3,3,تأسسات هاد لمنضمة نهار 14 شتنبر 1960 ف بغداد و...,"[تأسسات, هاد, لمنضمة, نهار, 14, شتنبر, 1960, ف...","[O, O, O, O, B-MISC, I-MISC, I-MISC, O, B-LOC,...",Morocco
4,4,ف 1965 ، رحلات لمنضمة ، و ولأّ لمقر ديالها ف ڤ...,"[ف, 1965, ،, رحلات, لمنضمة, ،, و, ولأّ, لمقر, ...","[O, O, O, O, O, O, O, O, O, O, O, B-LOC, O, O,...",Morocco
...,...,...,...,...,...
2506,2506,النشيد ديال لفيفا هو نشيد صاوب لألماني فرانز ل...,"[النشيد, ديال, لفيفا, هو, نشيد, صاوب, لألماني,...","[O, O, B-ORG, O, O, O, B-MISC, B-PER, I-PER, O...",Morocco
2507,2507,كايتغنى النشيد فلميطشان والتورنوايات اللي كاتن...,"[كايتغنى, النشيد, فلميطشان, والتورنوايات, اللي...","[O, O, O, O, O, O, B-ORG, O]",Morocco
2508,2508,من 2007 فرضات لفيفا على الشوركة دياولها اللي ك...,"[من, 2007, فرضات, لفيفا, على, الشوركة, دياولها...","[O, O, O, B-ORG, O, O, O, O, O, O, O, O, O, O,...",Morocco
2509,2509,كاينين ستة ديال لكونفيديراليات ولمنضامات الدول...,"[كاينين, ستة, ديال, لكونفيديراليات, ولمنضامات,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]",Morocco


In [68]:
dar_ner['Dialect'].value_counts()

Dialect
Morocco                   2177
Tunisia                    133
Algeria                     79
Oman                        49
Saudi Arabia                12
Egypt                       11
Libya                       10
Lebanon                      8
Jordan                       8
Sudan                        7
Syria                        7
Iraq                         6
Yemen                        2
Modern Standard Arabic       1
Qatar                        1
Name: count, dtype: int64

In [69]:
# Morocco
if not os.path.exists('data_dialects/Probing/NER/Morocco/DarNERcorp'):
    os.mkdir('data_dialects/Probing/NER/Morocco/DarNERcorp')
dar_ner_morocco = dar_ner[dar_ner['Dialect'] == 'Morocco']
train, test = train_test_split(dar_ner_morocco, test_size=0.2, random_state=42)
with open('data_dialects/Probing/NER/Morocco/DarNERcorp/dar_ner_morocco_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/Morocco/DarNERcorp/dar_ner_morocco_label_train.txt', 'w') as file:
    for tags in train['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/NER/Morocco/DarNERcorp/dar_ner_morocco_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/Morocco/DarNERcorp/dar_ner_morocco_label_test.txt', 'w') as file:
    for tags in test['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")

## CLEANANERcorp

MSA

In [70]:
with open('data/Probing/CLEANANERcorp/cleananercorp_TRAIN.txt', 'r') as file:
    cleananercorp_TRAIN = file.read()
with open('data/Probing/CLEANANERcorp/cleananercorp_TEST.txt', 'r') as file:
    cleananercorp_TEST = file.read()
cleananer = cleananercorp_TRAIN + '\n' + cleananercorp_TEST

final_data = []
for sentence in cleananer.split('\n\n'):
    tokens = []
    tags = []
    for line in sentence.split('\n'):
        if line.strip() == '':
            continue
        if len(line.split()) == 2:
            token, tag = line.split()
            tokens.append(token)
            tags.append(tag)
    if len(tokens) > 0:
        final_data.append({'Tokens': tokens, 'NER_Tags': tags})
cleananer = pd.DataFrame(final_data)
cleananer['sentence'] = cleananer['Tokens'].apply(lambda x: ' '.join(x))
cleananer['Sentence_ID'] = range(1, len(cleananer) + 1)
cleananer = cleananer[['Sentence_ID', 'sentence', 'Tokens', 'NER_Tags']]
cleananer['Dialect'] = identify_dialect_camel(cleananer['sentence'].tolist())
cleananer.to_csv('data/Probing/CLEANANERcorp/cleananer_cleaned.csv', index=False)
cleananer

,Sentence_ID,sentence,Tokens,NER_Tags,Dialect
0,1,فرانكفورت (د ب أ) أعلن اتحاد صناعة السيارات في...,"[فرانكفورت, (د, ب, أ), أعلن, اتحاد, صناعة, الس...","[B-LOC, B-ORG, I-ORG, I-ORG, O, B-ORG, I-ORG, ...",Oman
1,2,وقال رئيس الاتحاد برند جوتشولك عند إعلان آخر ت...,"[وقال, رئيس, الاتحاد, برند, جوتشولك, عند, إعلا...","[O, O, O, B-PERS, I-PERS, O, O, O, O, O, O, O,...",Modern Standard Arabic
2,3,وعلي الرغم من أنه قال أنه يتوقع أن تظل صادرات ...,"[وعلي, الرغم, من, أنه, قال, أنه, يتوقع, أن, تظ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Modern Standard Arabic
3,4,ورأي جوتشولك أنه يتعين أن يبلغ الحجم الاجمالي ...,"[ورأي, جوتشولك, أنه, يتعين, أن, يبلغ, الحجم, ا...","[O, B-PERS, O, O, O, O, O, O, O, O, O, O, O, O...",Modern Standard Arabic
4,5,وأضاف قائلا نادرا ما كان من الصعب التكهن بالنس...,"[وأضاف, قائلا, نادرا, ما, كان, من, الصعب, التك...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]",Modern Standard Arabic
...,...,...,...,...,...
4894,4895,""" الأمير السعودي في قطاعات استثمارية متباينة ت...","["", الأمير, السعودي, في, قطاعات, استثمارية, مت...","[O, O, B-MISC, O, O, O, O, O, O, O, O, O, O, B...",Modern Standard Arabic
4895,4896,وتتهم بن طلال شهادة البكالوريوس في إدارة الأعم...,"[وتتهم, بن, طلال, شهادة, البكالوريوس, في, إدار...","[O, B-PERS, I-PERS, O, O, O, O, O, O, O, O, O,...",Modern Standard Arabic
4896,4897,ذلك ، يتابع الوليد بن طلال أعماله التجارية عقب...,"[ذلك, ،, يتابع, الوليد, بن, طلال, أعماله, التج...","[O, O, O, B-PERS, I-PERS, B-PERS, O, O, O, O, ...",Modern Standard Arabic
4897,4898,الوزيرة ابن وابنة ، خالد وريم .,"[الوزيرة, ابن, وابنة, ،, خالد, وريم, .]","[O, O, O, O, B-PERS, B-PERS, O]",Tunisia


In [71]:
cleananer['Dialect'].value_counts()

Dialect
Modern Standard Arabic    3050
Oman                       584
Saudi Arabia               475
Iraq                       156
Sudan                      153
Libya                      110
Morocco                     76
Algeria                     74
Tunisia                     72
Egypt                       50
Yemen                       41
Jordan                      27
Syria                       20
Qatar                        6
Lebanon                      3
Palestine                    2
Name: count, dtype: int64

In [ ]:
# MSA
if not os.path.exists('data_dialects/Probing/NER/MSA/CLEANANERcorp'):
    os.mkdir('data_dialects/Probing/NER/MSA/CLEANANERcorp')
cleananer_msa = cleananer[cleananer['Dialect'] == 'Modern Standard Arabic']
train, test = train_test_split(cleananer_msa, test_size=0.2, random_state=42)
with open('data_dialects/Probing/NER/MSA/CLEANANERcorp/cleananer_msa_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/MSA/CLEANANERcorp/cleananer_msa_label_train.txt', 'w') as file:
    for tags in train['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/NER/MSA/CLEANANERcorp/cleananer_msa_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/MSA/CLEANANERcorp/cleananer_msa_label_test.txt', 'w') as file:
    for tags in test['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")

## MADAR

Oman

In [4]:
madar_oman = pd.read_csv("data/CKA/MADAR_Corpus/MADAR.corpus.Muscat.tsv", sep="\t")
madar_oman = madar_oman[['sent']].rename(columns={'sent': 'sentence'})
madar_oman.reset_index(drop=True, inplace = True)


from camel_tools.ner import NERecognizer
from camel_tools.tokenizers.word import simple_word_tokenize
ner = NERecognizer('CAMeL-Lab/bert-base-arabic-camelbert-da-ner', use_gpu=True)

madar_oman['Tokens'] = madar_oman['sentence'].apply(lambda x: simple_word_tokenize(x))
madar_oman['NER_Tags'] = ner.predict(madar_oman['Tokens'].tolist())
madar_oman['Sentence_ID'] = range(1, len(madar_oman) + 1)
madar_oman['sentence'] = madar_oman['Tokens'].apply(lambda x: ' '.join(x))
madar_oman = madar_oman[['Sentence_ID', 'sentence', 'Tokens', 'NER_Tags']]

# ner_tags = madar_oman[['Tokens', 'NER_Tags']].explode(['Tokens','NER_Tags'])
# ner_tags.reset_index(inplace=True)
# ner_tags.rename(columns={'index': 'Sentence_ID'}, inplace=True)
# ner_tags['Sentence_ID'] = ner_tags['Sentence_ID'] + 1


Some weights of the model checkpoint at CAMeL-Lab/bert-base-arabic-camelbert-da-ner were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [5]:
# Oman
if not os.path.exists('data_dialects/Probing/NER/Oman/MADAR'):
    os.mkdir('data_dialects/Probing/NER/Oman/MADAR')
train, test = train_test_split(madar_oman, test_size=0.2, random_state=42)
with open('data_dialects/Probing/NER/Oman/MADAR/madar_oman_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/Oman/MADAR/madar_oman_label_train.txt', 'w') as file:
    for tags in train['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/NER/Oman/MADAR/madar_oman_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/Oman/MADAR/madar_oman_label_test.txt', 'w') as file:
    for tags in test['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")

## MADAR

Saudi

In [6]:
madar_saudi = pd.read_csv("data/CKA/MADAR_Corpus/MADAR.corpus.Riyadh.tsv", sep="\t")
madar_saudi = madar_saudi[['sent']].rename(columns={'sent': 'sentence'})
madar_saudi.reset_index(drop=True, inplace = True)


from camel_tools.ner import NERecognizer
from camel_tools.tokenizers.word import simple_word_tokenize
ner = NERecognizer('CAMeL-Lab/bert-base-arabic-camelbert-da-ner', use_gpu=True)

madar_saudi['Tokens'] = madar_saudi['sentence'].apply(lambda x: simple_word_tokenize(x))
madar_saudi['NER_Tags'] = ner.predict(madar_saudi['Tokens'].tolist())
madar_saudi['Sentence_ID'] = range(1, len(madar_saudi) + 1)
madar_saudi['sentence'] = madar_saudi['Tokens'].apply(lambda x: ' '.join(x))
madar_saudi = madar_saudi[['Sentence_ID', 'sentence', 'Tokens', 'NER_Tags']]

# ner_tags = madar_saudi[['Tokens', 'NER_Tags']].explode(['Tokens','NER_Tags'])
# ner_tags.reset_index(inplace=True)
# ner_tags.rename(columns={'index': 'Sentence_ID'}, inplace=True)
# ner_tags['Sentence_ID'] = ner_tags['Sentence_ID'] + 1


Some weights of the model checkpoint at CAMeL-Lab/bert-base-arabic-camelbert-da-ner were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [7]:
# Saudi Arabia
if not os.path.exists('data_dialects/Probing/NER/Saudi/MADAR'):
    os.mkdir('data_dialects/Probing/NER/Saudi/MADAR')
train, test = train_test_split(madar_saudi, test_size=0.2, random_state=42)
with open('data_dialects/Probing/NER/Saudi/MADAR/madar_saudi_sentence_train.txt', 'w') as file:
    for sent in train['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/Saudi/MADAR/madar_saudi_label_train.txt', 'w') as file:
    for tags in train['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")
with open('data_dialects/Probing/NER/Saudi/MADAR/madar_saudi_sentence_test.txt', 'w') as file:
    for sent in test['sentence'].tolist():
        file.write(f"{sent}\n")
with open('data_dialects/Probing/NER/Saudi/MADAR/madar_saudi_label_test.txt', 'w') as file:
    for tags in test['NER_Tags'].tolist():
        file.write(f"{' '.join(tags)}\n")